In [1]:
from graph.graph_builder import FolktaleGraphBuilder
from utils.loader import load_json
from schemas.folktale import Folktale
from graph.neo4j_manager import Neo4jManager
from tqdm import tqdm
import os


In [2]:
out_dir = "./out2"


In [3]:
folktales = []
for file in os.listdir(out_dir):
	path = os.path.join(out_dir, file)
	folktale = load_json(path)
	folktales.append(Folktale(**folktale))


In [4]:
neo4j = Neo4jManager()


In [5]:
graph_builder = FolktaleGraphBuilder(neo4j)


In [6]:
graph_builder.clear_database()
graph_builder.setup()


In [7]:
for folktale in tqdm(folktales):
	graph_builder.add_folktale(folktale)
	

100%|██████████| 5/5 [00:11<00:00,  2.34s/it]


In [8]:
graph_builder.find_loose_entities()


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL {\n            MATCH (o:Object)\n            WHERE NOT EXISTS {\n                MATCH (o)<-[:HAS_OBJECT]-(:Event)\n            }\n            RETURN 'Object' AS type, o.id AS id, o.name AS name,\n                'No event link' AS reason, o AS node\n\n            UNION ALL\n\n            MATCH (a:Agent)\n            WHERE NOT EXI

[{'type': 'Object',
  'id': 'object_ac7ad96421b24657826fb77e4e46d923',
  'name': 'horse',
  'reason': 'No event link'},
 {'type': 'Place',
  'id': 'place_8310f53cfbf546e0a8c35ae5bfd452f7',
  'name': 'The Village',
  'reason': 'No event usage and unlinked'},
 {'type': 'Place',
  'id': 'place_f7617767a55c4e34ad87e88cffb7e912',
  'name': 'The Farm Ground',
  'reason': 'No event usage and unlinked'}]

In [9]:
graph_builder.delete_loose_entities()


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL {\n            MATCH (o:Object)\n            WHERE NOT EXISTS { MATCH (o)<-[:HAS_OBJECT]-(:Event) }\n            WITH collect(DISTINCT o) AS nodes\n            WITH nodes, [n IN nodes | n.name] AS names\n            UNWIND nodes AS o\n            DETACH DELETE o\n            RETURN 'Object' AS type, names\n\n            UNION ALL\

[{'type': 'Object', 'names': ['horse']},
 {'type': 'Place', 'names': ['The Village', 'The Farm Ground']},
 {'type': 'Place', 'names': ['The Village', 'The Farm Ground']}]

In [10]:
graph_builder.find_loose_entities()


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL {\n            MATCH (o:Object)\n            WHERE NOT EXISTS {\n                MATCH (o)<-[:HAS_OBJECT]-(:Event)\n            }\n            RETURN 'Object' AS type, o.id AS id, o.name AS name,\n                'No event link' AS reason, o AS node\n\n            UNION ALL\n\n            MATCH (a:Agent)\n            WHERE NOT EXI

[]

In [11]:
graph_builder.create_vector_databases()


El índice vectorial 'event_embeddings' ya existe. Saltando.
El índice vectorial 'agent_embeddings' ya existe. Saltando.
El índice vectorial 'place_embeddings' ya existe. Saltando.
El índice vectorial 'object_embeddings' ya existe. Saltando.


In [12]:
neo4j.close()
